<a href="https://colab.research.google.com/drive/1W-Jzy30ghlSbIzzgrwcIjiV3_AX4pI5S?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

### Self-Evolving Agent

In [ ]:
!pip install -qU google-genai

In [ ]:
from google import genai
import getpass

Get free-tier Google's Gemini API Key here: https://aistudio.google.com/app/apikey

In [ ]:
API_KEY = getpass.getpass("Enter your Google API key: ")

In [ ]:
client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-flash-latest"

In [ ]:
import re

In [ ]:
class SelfEvolvingAgent:
    """An agent that rewrites its own instructions between attempts.

    Reflexion critiques an answer. This critiques the *rulebook* that produced
    the answer, so what improves is carried into the next task rather than
    thrown away with the last one. The rulebook is the only thing that
    persists; the attempts themselves are discarded.
    """

    def __init__(self, standard):
        self.model = MODEL_NAME
        self.rules = ["Answer the task directly."]
        self.history = []
        # The bar the critic holds. The author never sees it - it has to be
        # discovered through the rulebook, which is the whole point. If the
        # author could read the standard, this would just be a longer prompt.
        self.standard = standard

    def rulebook(self):
        return "\n".join(f"- {rule}" for rule in self.rules)

    def attempt(self, task):
        prompt = f"""You are solving a task. Follow every rule below exactly.

        Rules:
        {self.rulebook()}

        Task: {task}

        Answer:"""

        return client.models.generate_content(
            model=self.model, contents=prompt
        ).text.strip()

    def critique(self, task, answer):
        """Judge the answer, then propose ONE new durable rule."""
        prompt = f"""You are reviewing an agent's work.

        Task: {task}

        The agent's rules were:
        {self.rulebook()}

        The agent answered:
        {answer}

        Judge it against this standard, which the agent cannot see:
        {self.standard}

        Reply in exactly this format and nothing else:

        VERDICT: PASS or FAIL
        WEAKNESS: one sentence, or "none"
        RULE: one new instruction that would prevent this weakness on ANY
        future task of this kind, phrased generally, or "none"

        Do not quote the standard back verbatim and do not mention that a
        standard exists - write the rule as general working advice. A rule
        that names this specific task is useless. Only say PASS if the answer
        already meets every part of the standard."""

        text = client.models.generate_content(
            model=self.model, contents=prompt
        ).text

        def field(name):
            match = re.search(rf"{name}:\s*(.+)", text)
            return match.group(1).strip() if match else ""

        return field("VERDICT").upper().startswith("PASS"), field("WEAKNESS"), field("RULE")

    def learn(self, rule):
        """Adopt a rule, unless it is empty or already known.

        Without this guard the rulebook grows a near-duplicate every round
        and the prompt fills with restatements of the same instruction.
        """
        rule = rule.strip().rstrip(".")
        if not rule or rule.lower() in {"none", "n/a"}:
            return False
        for existing in self.rules:
            if rule.lower() in existing.lower() or existing.lower().rstrip(".") in rule.lower():
                return False
        self.rules.append(rule + ".")
        return True

    def solve(self, task, max_rounds=3):
        print(f"\nTask: {task}")
        answer = ""

        for round_number in range(1, max_rounds + 1):
            answer = self.attempt(task)
            passed, weakness, rule = self.critique(task, answer)

            print(f"\n  Round {round_number}: {'PASS' if passed else 'FAIL'}")
            print(f"  Answer: {answer[:200]}{'...' if len(answer) > 200 else ''}")

            if passed:
                break

            print(f"  Weakness: {weakness}")
            if self.learn(rule):
                print(f"  Learned:  {rule}")
            else:
                # No new rule means another round would run the same agent
                # against the same task, so stop rather than burn a call.
                print("  No new rule proposed; stopping.")
                break

        self.history.append({"task": task, "answer": answer, "rules": len(self.rules)})
        return answer

In [ ]:
# Three unrelated tasks, one agent. Watch the rulebook carry over.
print("=" * 60)
print("A SINGLE AGENT ACROSS THREE TASKS")
print("=" * 60)
print("The agent starts with one rule and is never shown the standard it is")
print("being judged against. Watch what it works out.")

# The critic holds a bar the author was never told about.
STANDARD = (
    "The answer must be under 60 words, must contain one concrete example "
    "with real numbers or names, and must end with a sentence naming what "
    "goes wrong when this is ignored."
)

agent = SelfEvolvingAgent(STANDARD)

agent.solve("Explain what a database index is.")
agent.solve("Explain what a race condition is.")
agent.solve("Explain what a memory leak is.")

In [ ]:
# What the agent taught itself
print("=" * 60)
print("THE RULEBOOK IT ENDED WITH")
print("=" * 60)
for i, rule in enumerate(agent.rules, 1):
    print(f"{i}. {rule}")

print()
print("Rules at the start: 1")
print(f"Rules at the end:   {len(agent.rules)}")
print()
for record in agent.history:
    print(f"  after {record['task'][:45]:45s} -> {record['rules']} rules")
print()
print("The standard the critic was holding all along:")
print(f"  {STANDARD}")
print()
print("If the rules above resemble that standard, the agent recovered it from")
print("nothing but repeated failure - it was never given the text.")

In [ ]:
# The part this pattern does not solve
#
# Nothing above verifies that a learned rule is *correct*. The critic and the
# author are the same model, so a confident mistake can be promoted into the
# rulebook and then applied to every future task - the failure mode is a
# system that gets steadily more consistent about being wrong.
#
# Two cheap guards are worth knowing:
#
#   1. Keep the rulebook bounded. It enters every prompt, so unbounded growth
#      is unbounded cost, and late rules start contradicting early ones.
#   2. Score the rulebook, not the answer. Re-run a fixed set of tasks after
#      each change and keep the rulebook only if the score did not drop. That
#      turns "the model liked it" into a measurement.

print(f"Rulebook size: {len(agent.rules)} rules, "
      f"{sum(len(r) for r in agent.rules)} characters carried into every prompt.")